The goal of this notebook is to train some adversarial patches on the ImageNet data set and visualize what these patches look like, as well as explore the effectiveness of patches once we alter certain features.

This notebook is largely modelled after: https://github.com/AIPI-590-XAI/Duke-AI-XAI/blob/main/adversarial-ai-example-notebooks/adversarial_attacks_patches.ipynb

I am writing my own version of tihs notebook to build a stronger understanding of this work.

In [1]:
# Install necessary packages if they do not exist.
# %pip install --upgrade numpy
# %pip install numpy matplotlib torch torchvision
# %pip install --upgrade pip

In [2]:
import os
import numpy as np
import math

import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import datasets, transforms

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")


GPU available: False


There exist many kinds of models that torchvision offers. In terms of image classification, ResNet50 is mainstream. Choosing default pre-trained weights for ResNet50 forces the most up-to-date version.

In [3]:
pretrained_model = torchvision.models.resnet50(weights = ResNet50_Weights.DEFAULT)
pretrained_model.to(device) # moves model to device

pretrained_model.eval()
for p in pretrained_model.parameters():
    p.requires_grad = False

We need to normalize input. Color channel intensity is measured by z scores as opposed to their actual values. 

Numbers for mean and standard deviation can be found here: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html

In [4]:
# Mean and Std from ImageNet
NORM_MEAN = np.array([0.485, 0.456, 0.406])
NORM_STD = np.array([0.229, 0.224, 0.225])


plain_transforms = transforms.Compose([
    transforms.ToTensor(), # Maps RGB to (0,1) as opposed to (0,255)
    transforms.Normalize(mean=NORM_MEAN,
                         std=NORM_STD) # 
])



We first want to evaluate how our model performs on unperturbed images. TinyImageNet contains 200 separate labels, each with 5 images which are drawn from ImageNet. ResNet50's pretrained weights are trained on IMAGENET1K_V2 (as of October 13th, 2026)

In [5]:
imagenet_path = "./data/TinyImageNet"

dataset = datasets.ImageFolder(
    root = imagenet_path,
    transform = plain_transforms
    )

data_loader = DataLoader(
    dataset,
    batch_size = 32,
    shuffle = True,
    drop_last = False
)

print("Images:", len(dataset))
print("Classes:", len(dataset.classes))

images, labels = next(iter(data_loader))
print("Batch shape:", images.shape)

Images: 5000
Classes: 1000
Batch shape: torch.Size([32, 3, 224, 224])


Now let's evaluate the model:

In [6]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct_top1 = 0
    correct_top5 = 0
    total = 0


    with torch.no_grad():
        for images, labels in data_loader:
            # move image and label to GPU
            images = images.to(device)
            labels = labels.to(device)

            # inputs a image batch of 32. Returns [batchsize, 1000].
            # Each entry in 1000 represents the logits that our model attributes.
            scores = model(images) 

            # index of maximum value along horizontal axis, which is logits
            predictions = scores.argmax(dim = 1) 
            # item() returns the number into a python integer as opposed to a tensor num
            correct_top1 += (predictions == labels).sum().item() 

            # finds top 5 scoring indices
            top5_predictions = scores.topk(5, dim=1).indices 
            correct_top5 += (
                top5_predictions == labels.unsqueeze(dim = 1)
            ).any(dim = 1).sum().item()

            total += labels.size(0)

    return correct_top1 / total, correct_top5 / total

top1, top5 = evaluate_model(pretrained_model, data_loader, device)
error1 = 1 - top1
error5 = 1 - top5
print(f"Top-1 Accuracy {top1:.2%}")
print(f"Top-5 Accuracy {top5:.2%}")

print(f"Top-1 Error {error1:.2%}")
print(f"Top-5 Error {error5:.2%}")

Top-1 Accuracy 86.76%
Top-5 Accuracy 98.04%
Top-1 Error 13.24%
Top-5 Error 1.96%


Let's show some images to see what our model is doing before we begin training some patches.

In [7]:
from matplotlib.ticker import PercentFormatter

class_names = ResNet50_Weights.DEFAULT.meta["categories"]

def show_predictions(model, data_loader, device, num_images=4):
    model.eval()
    images, labels = next(iter(data_loader))
    num_images = min(num_images, images.size(0)) # displays as many images as pos

    # get the images and labels, move to cpu since matplotlib works on cpu
    images = images[:num_images].cpu()
    labels = labels[:num_images].cpu()

    # disable tracking of how much our output changes
    with torch.no_grad():
        # move image to device, then process model prediction. shape num_images x num_labels
        logits = model(images.to(device))
        probabilities = logits.softmax(dim=1)
        top_probs, top_indices = probabilities.topk(5, dim=1)

        # moves our work back to cpu
        top_probs = top_probs.cpu()
        top_indices = top_indices.cpu()

        # what exactly is this doing?
        mean = images.new_tensor(NORM_MEAN).view(1,3,1,1)
        std = images.new_tensor(NORM_STD).view(1,3,1,1)

        display_images = (images * std + mean).clamp(0, 1) # undo normalization constants

        fix, axes = plt.subplots(
            nrows = num_images,
            ncols = 2,
            figsize= (12, 3 * num_images),
            squeeze = False,
            constrained_layout = True
        )

        for i in range(num_images):
            image_ax, graph_ax = axes[i]
            image_ax.imshow(display_images[i].permute(1,2,0).numpy()) # reorder dimensions
            true_label = data_loader.dataset.classes[labels[i].item()]
            predicted_label = class_names[top_indices[i, 0].item()]

            image_ax.set_title(
                f"Dataset Label: {true_label}\n"
                f"Prediction: {predicted_label}" 
            )

            image_ax.axis("off")

            # look at the corresponding name for each prediction
            names = [class_names[index.item()] for index in top_indices[i]]
            values = top_probs[i].numpy()
            graph_ax.barh(range(5), values, color="steelblue") # horizontal bars
            graph_ax.set_yticks(range(5), labels=names)
            graph_ax.invert_yaxis() # 0 first
            graph_ax.set_xlim(0, 1)
            graph_ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

            graph_ax.set_xlabel("Predicted probability")
            graph_ax.set_title("Top five predictions")

        plt.show()



In [8]:
# show_predictions(pretrained_model, data_loader, device, num_images=4)

**What patch training will do**


We choose a target such as toaster. One patch is shared across many background images. The goal is to increase ResNet50's probability for that target whenever the patch is present.

The patch is an nn.Parameter shaped [3, patch_height, patch_width]. Its entries are unrestricted trainable numbers, not a separate neural network. For a 64-by-64 patch, there are 3 * 64 * 64 = 12,288 trainable numbers.

Before inserting the patch, convert each parameter \(u\) into an RGB intensity:

P(U) = (tanh(u) + 1)/2

tanh smoothly maps real numbers into (-1, 1), and the shift and division map them into (0, 1) (floating-point arithmetic can reach endpoints). Zero parameters therefore produce RGB intensity 0.5: a gray patch. The RGB intensities are then normalized with the same channel constants as the background images.

For every batch, insert the same patch at an independently sampled location in each image. The placement function clones the image tensor first, so the original batch remains available and repeated evaluation does not accumulate extra patches.

Run the patched images through the frozen model to obtain logits shaped [batch_size, 1000]. Every optimization target is the selected ImageNet class index. CrossEntropyLoss accepts logits and internally performs the log-softmax calculation. For one image, its loss is \(-\log(p_{target})\). If target probability increases from 0.01 to 0.9, this loss falls from about 4.605 to 0.105.

loss.backward() applies the chain rule through ResNet, placement, normalization, and tanh, storing a derivative for every patch parameter in patch.grad. The optimizer receives [patch], so only that parameter tensor can be updated. We do not place the training forward pass inside torch.no_grad().

SGD uses learning rate 0.1 and momentum 0.8. Gradients from the images in a batch are averaged by the loss. Several epochs expose the evolving patch to different backgrounds and locations. Each epoch prints sample-weighted training loss and mean target probability; those are training statistics, not held-out attack success rates.

A 90/10 image split keeps a small set of backgrounds out of optimization. The later clean/patched comparison uses those held-out backgrounds. These cells vary only position: they do not establish robustness to rotation, lighting, printing, or occlusion. The held-out visualization includes all selected images regardless of their true class and is not a strict targeted attack success rate.

In [9]:
import torch.nn as nn
from torch.utils.data import random_split

# Fix the random sequence for this experiment. This does not guarantee
# identical results across different devices or PyTorch versions.
torch.manual_seed(42)

if len(dataset) < 2:
    raise ValueError("At least two images are needed for a train/validation split.")

validation_size = max(1, int(0.1 * len(dataset))) # 10% of the data set
training_size = len(dataset) - validation_size # 90%
patch_train_set, patch_val_set = random_split(
    dataset,
    [training_size, validation_size],
    generator=torch.Generator().manual_seed(42),
)

# Keep your existing data_loader unchanged. These loaders serve patch training.
# num_workers defaults to 0 for compatibility across local notebooks and Colab.
patch_train_loader = DataLoader(patch_train_set, batch_size=32, shuffle=True)
patch_val_loader = DataLoader(patch_val_set, batch_size=32, shuffle=False)

print(f"Patch training images: {training_size}")
print(f"Held-out images: {validation_size}")

Patch training images: 4500
Held-out images: 500


Convert parameters into colors, then place the patch
Keep these three representations distinct: patch stores trainable parameters; patch_to_rgb(patch) returns displayable RGB intensities; normalize_patch(patch) returns values compatible with the normalized model input. Reuse the same conversion in training and visualization.

In [10]:
def patch_to_rgb(patch):
    # This remains connected to the trainable parameter through autograd.
    # Increasing a parameter increases the corresponding RGB channel intensity.
    return (torch.tanh(patch) + 1) / 2

def normalize_patch(patch):
    rgb = patch_to_rgb(patch)  # [3, patch_height, patch_width]

    # [3, 1, 1] broadcasts each channel's constant across its entire plane.
    # new_tensor matches the patch's device and floating-point dtype.
    mean = rgb.new_tensor(NORM_MEAN).view(3, 1, 1)
    std = rgb.new_tensor(NORM_STD).view(3, 1, 1)
    return (rgb - mean) / std

Place the patch

In [11]:
def place_patch(images, patch):
    # images: [batch_size, 3, image_height, image_width], already normalized.
    # patch:  [3, patch_height, patch_width], unrestricted trainable parameters.
    if images.device != patch.device:
        raise ValueError("Images and patch must be on the same device.")

    _, _, image_height, image_width = images.shape # batch size, rgb, height, width
    _, patch_height, patch_width = patch.shape #rgb, height, width.
    if patch_height > image_height or patch_width > image_width:
        raise ValueError("The patch must fit inside the image.")

    # Clone preserves the original images and still supports autograd.
    patched_images = images.clone()
    normalized_patch = normalize_patch(patch)

    # place at random location for each patch in the batch
    for i in range(images.size(0)):
        # randint excludes its upper bound. The +1 includes the bottom/right
        # edge positions and allows a patch exactly as large as the image.
        top = torch.randint(0, image_height - patch_height + 1, (1,)).item()
        left = torch.randint(0, image_width - patch_width + 1, (1,)).item()

        # Replace all three channels in this rectangle with the SAME patch.
        # Slice assignment preserves the gradient path to the patch parameter.
        patched_images[
            i, :, top:top + patch_height, left:left + patch_width
        ] = normalized_patch

    return patched_images

**Train the shared patch**

The dataset's original labels are ignored here: the optimization label for every image is the selected target class. zero_grad() clears the previous derivative, backward() calculates the current derivative, and step() changes the patch. The patch is initialized only once per function call, so it accumulates learning across batches and epochs.

In [12]:
def train_patch(model, train_loader, target_index, device,
                patch_size=64, epochs=5, learning_rate=0.1):
    if patch_size < 1 or epochs < 1:
        raise ValueError("patch_size and epochs must be positive integers.")

    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    # All zeros become RGB 0.5 after patch_to_rgb(): the initial patch is gray.
    # Create it on the final device BEFORE constructing the optimizer.
    patch = nn.Parameter(torch.zeros(3, patch_size, patch_size, device=device))

    # Only this tensor is registered with the optimizer, not model.parameters().
    optimizer = torch.optim.SGD([patch], lr=learning_rate, momentum=0.8)
    criterion = nn.CrossEntropyLoss()  # Mean loss over the current batch.
    history = {"loss": [], "target_probability": []}

    for epoch in range(epochs):
        loss_sum = 0.0
        target_probability_sum = 0.0
        image_count = 0

        for images, _ in train_loader:
            images = images.to(device)
            optimizer.zero_grad(set_to_none=True)

            patched_images = place_patch(images, patch)
            # Keep autograd enabled: we need the model's INPUT derivatives.
            logits = model(patched_images)  # [batch_size, 1000]

            # These are target logits
            targets = torch.full(
                (images.size(0),), target_index,
                dtype=torch.long, device=device,
            )
            loss = criterion(logits, targets)  # Pass logits, not softmax values.
            loss.backward()  # Populates patch.grad using the chain rule.
            optimizer.step()  # Updates patch values and its momentum buffer.

            # Detach reporting values so they are not tracked for gradients.
            # These predictions were made BEFORE the current optimizer update.
            batch_size = images.size(0)
            loss_sum += loss.item() * batch_size
            target_probability_sum += (
                logits.detach().softmax(dim=1)[:, target_index].sum().item()
            )
            image_count += batch_size

        if image_count == 0:
            raise ValueError("The training loader did not yield any images.")

        # Weight by image count so a smaller final batch is handled correctly.
        mean_loss = loss_sum / image_count
        mean_probability = target_probability_sum / image_count
        history["loss"].append(mean_loss)
        history["target_probability"].append(mean_probability)
        print(
            f"Epoch {epoch + 1}/{epochs} | Loss: {mean_loss:.3f} | "
            f"Mean training target probability: {mean_probability:.1%}"
        )

    # Return PARAMETERS, not RGB values. The same patch_to_rgb conversion is
    # required for display. CPU output is convenient for plotting and saving.
    return patch.detach().cpu(), history


**Run one experiment and inspect its appearance**

Start with one epoch to confirm that the cells work in your environment, then rerun with five epochs. Calling train_patch again initializes a new patch; it does not resume the previous run. The target class is looked up in ResNet's own 1,000-category list, not in the dataset's folder list.

In [ ]:
class_names = ResNet50_Weights.IMAGENET1K_V2.meta["categories"]
target_name = "toaster"
target_index = class_names.index(target_name)

patch, history = train_patch(
    pretrained_model,
    patch_train_loader,
    target_index,
    device,
    patch_size=64,
    epochs=1,
)

rgb_patch = patch_to_rgb(patch).permute(1, 2, 0).numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].imshow(rgb_patch)
axes[0].set_title(f"Learned patch: {target_name}")
axes[0].axis("off")

epoch_numbers = range(1, len(history["loss"]) + 1)
axes[1].plot(epoch_numbers, history["loss"], marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean training cross entropy")
axes[1].set_xticks(list(epoch_numbers))
axes[1].set_title("Patch training loss")
plt.show()
